# Entrenamiento YOLOv8 — Detección de Aves

**Proyecto:** Visión Artificial — Reconocimiento de Aves con YOLO

**Autor:** Christian

**Institución:** CETI — Mecatrónica, 6to Semestre

Este notebook está pensado para correrse en **Google Colab** con GPU activada
(`Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`).

Pasos que cubre:
1. Instalación de dependencias
2. Descarga del dataset de aves (Roboflow)
3. Entrenamiento del modelo YOLOv8
4. Evaluación de métricas (mAP, precisión, recall)
5. Inferencia / pruebas y generación de evidencias con bounding boxes

## 1. Instalación de dependencias

In [ ]:
!pip install ultralytics roboflow -q

import ultralytics
ultralytics.checks()

## 2. Descarga del dataset (Roboflow)

Dataset usado: **"birds" by kurisnis** — 277 imágenes anotadas, clase única `bird`,
formato YOLOv8. Disponible públicamente en Roboflow Universe:

https://universe.roboflow.com/kurisnis/birds-olyak

> **Nota:** Necesitas una API key gratuita de Roboflow (te la dan al crear cuenta en
> https://roboflow.com). Pégala donde dice `YOUR_API_KEY`.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("kurisnis").project("birds-olyak")
version = project.version(1)
dataset = version.download("yolov8")

print("Dataset descargado en:", dataset.location)

### Alternativa sin cuenta de Roboflow

Si no quieres crear una API key, puedes descargar el ZIP del dataset directamente
desde la página del proyecto (botón "Download Dataset" > formato `YOLOv8`) y subirlo
a Colab, o usar `!wget` con el link de descarga directa que te genera Roboflow.
Descomprímelo en una carpeta llamada `birds-olyak-1/` con la siguiente estructura:

```
birds-olyak-1/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
├── test/
│   ├── images/
│   └── labels/
└── data.yaml
```

## 3. Revisar el archivo data.yaml

Este archivo define las clases y rutas que usará YOLO durante el entrenamiento.

In [ ]:
with open(f"{dataset.location}/data.yaml", "r") as f:
    print(f.read())

## 4. Entrenamiento del modelo

Usamos `yolov8n.pt` (versión *nano*) como punto de partida porque:
- Entrena rápido incluso en la GPU gratuita de Colab.
- Es suficiente para un dataset de una sola clase (~277 imágenes).
- Permite iterar rápido si hay que ajustar hiperparámetros.

Si tienes más tiempo/GPU disponible, puedes cambiar a `yolov8s.pt` o `yolov8m.pt`
para mejor precisión.

In [ ]:
from ultralytics import YOLO

# Cargamos el modelo base preentrenado en COCO
model = YOLO("yolov8n.pt")

# Entrenamiento
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,          # early stopping si no mejora
    project="runs_aves",
    name="yolov8n_aves",
    seed=42
)

## 5. Evaluación del modelo

Validamos el modelo entrenado sobre el set de validación y revisamos métricas
clave: **mAP50**, **mAP50-95**, **precisión** y **recall**.

In [ ]:
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precisión:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## 6. Inferencia sobre imágenes de prueba (generación de evidencias)

Corremos el modelo entrenado sobre el set de `test/` para generar imágenes
con bounding boxes. Estas imágenes son las que se suben a la carpeta
`evidencias/` del repositorio.

In [ ]:
import glob

test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")
print(f"Imágenes de prueba encontradas: {len(test_images)}")

# Tomamos el mejor modelo entrenado (best.pt)
best_model = YOLO("runs_aves/yolov8n_aves/weights/best.pt")

pred_results = best_model.predict(
    source=test_images[:15],   # primeras 15 imágenes de prueba
    conf=0.35,
    save=True,
    project="runs_aves",
    name="predicciones_evidencias"
)

print("Predicciones guardadas en: runs_aves/predicciones_evidencias/")

### Visualizar algunas predicciones directamente en el notebook

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

pred_imgs = glob.glob("runs_aves/predicciones_evidencias/*.jpg")[:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, img_path in zip(axes.flatten(), pred_imgs):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.axis("off")
plt.tight_layout()
plt.savefig("grid_evidencias.png", dpi=150)
plt.show()

## 7. Prueba con video (opcional)

Si tienes un video de aves, puedes correr detección frame por frame y exportar
un video anotado como evidencia adicional.

In [ ]:
# Descomenta y ajusta la ruta si tienes un video de prueba

# video_path = "/content/video_prueba_aves.mp4"
# best_model.predict(
#     source=video_path,
#     conf=0.35,
#     save=True,
#     project="runs_aves",
#     name="video_evidencia"
# )

## 8. Exportar el modelo entrenado

Guardamos los pesos finales (`best.pt`) para poder usarlos después sin reentrenar.

In [ ]:
import shutil

shutil.copy("runs_aves/yolov8n_aves/weights/best.pt", "modelo_aves_best.pt")
print("Modelo guardado como modelo_aves_best.pt")

# En Colab, descarga el archivo a tu computadora:
# from google.colab import files
# files.download("modelo_aves_best.pt")

## 9. Siguiente paso

1. Descarga las imágenes generadas en `runs_aves/predicciones_evidencias/` y
   colócalas en la carpeta `evidencias/` del repositorio.
2. Descarga la curva de resultados (`runs_aves/yolov8n_aves/results.png`) y
   también guárdala en `evidencias/` — sirve como evidencia del entrenamiento.
3. Haz commit y push de todo al repositorio de GitHub.